In [5]:
import base64
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Dict, Union

import requests

# =========================
# CONFIG (match Stage style)
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
MAX_TOKENS_TO_USE = 7

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 6
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60

# FULL_NAME = "behnamparsa/toDoList"
# RUN_ID = 20786203669

FULL_NAME = "FooIbar/EhViewer"
RUN_ID = 22557070276


# Where to save the workflow file text
OUT_PATH = Path(r"C:\Android Mobile App\ICST2026_Ext\log") / f"workflow_{RUN_ID}.yml"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# =========================
# Token loader (same as your stages)
# =========================
def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

# =========================
# Minimal GitHub client (like Stage code)
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "workflow-fetcher/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                return i
            if st.reset_epoch and st.reset_epoch <= now:
                return i
        return 0

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None
            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue
            if resp.status_code >= 400:
                print(f"[error] {resp.status_code} {url}")
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

# =========================
# Helpers: get run metadata then fetch workflow content at that SHA
# =========================
def get_run_metadata(gh: GitHubClient, full_name: str, run_id: int) -> Dict:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}"
    data = gh.request_json("GET", url, params=None)
    if not data or not isinstance(data, dict):
        raise RuntimeError("Failed to fetch run metadata.")
    return data

def fetch_workflow_yaml_at_ref(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    # Use the same endpoint your Stage 3 uses: /contents/<workflow_path>?ref=<sha>
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    # fallback to download_url
    dl = data.get("download_url")
    if dl:
        # signed/raw URL doesn’t need auth in many cases, but it can; keep auth session
        r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
        if r.status_code == 200:
            return r.text or ""
    return ""

# =========================
# MAIN
# =========================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    run = get_run_metadata(gh, FULL_NAME, RUN_ID)

    head_sha = (run.get("head_sha") or "").strip()
    path = (run.get("path") or "").strip()  # NOTE: sometimes empty
    workflow_url = run.get("workflow_url")  # points to workflow metadata

    # If run['path'] isn't present, derive workflow path from other fields:
    # Many runs include 'workflow_id' and 'workflow_url'; easiest is to read your own known path,
    # but we can fetch it from the run payload if provided.
    workflow_path = (run.get("workflow_path") or run.get("path") or "").strip()

    print("head_sha:", head_sha)
    print("workflow_path (from run payload):", workflow_path)
    print("workflow_url:", workflow_url)

    if not head_sha:
        raise SystemExit("No head_sha found in run payload; cannot fetch workflow at ref.")

    if not workflow_path:
        # You can hardcode if needed (you already know it for this sample)
        workflow_path = ".github/workflows/instru_test_GMD.yml"
        print("workflow_path missing in payload; using fallback:", workflow_path)

    yml = fetch_workflow_yaml_at_ref(gh, FULL_NAME, workflow_path, ref=head_sha)
    if not yml.strip():
        raise SystemExit("Fetched workflow content is empty. Check token scopes / path / ref.")

    OUT_PATH.write_text(yml, encoding="utf-8")
    print("\nSaved workflow YAML to:")
    print(str(OUT_PATH))

    # Optional: print extracted '- name:' lines for quick verification
    print("\nStep names found in fetched YAML:")
    for line in yml.splitlines():
        if line.lstrip().startswith("- name:"):
            print(line.strip())

if __name__ == "__main__":
    main()


head_sha: 66e43924bddcc773606aa231dc6a8fc4ccea5c76
workflow_path (from run payload): .github/workflows/baseline-profile.yml
workflow_url: https://api.github.com/repos/FooIbar/EhViewer/actions/workflows/104482674

Saved workflow YAML to:
C:\Android Mobile App\ICST2026_Ext\log\workflow_22557070276.yml

Step names found in fetched YAML:
- name: Free disk space
- name: Enable KVM group perms
- name: Checkout
- name: Setup Java
- name: Install Rust Toolchain
- name: Rust Cache
- name: CMake Cache
- name: Setup Gradle
- name: Generate Baseline Profile
- name: Upload reports


In [3]:
#steps name: Executed

In [4]:
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Dict, Union

import requests
import pandas as pd


# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
MAX_TOKENS_TO_USE = 7

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 6
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60

FULL_NAME = "FooIbar/EhViewer"

# This is the workflow run ID, not the job ID.
RUN_ID = 22557070276

OUT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\log")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / f"executed_steps_run_{RUN_ID}.csv"


# =========================
# Token loader
# =========================
def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []

    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")

        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)

            if len(tokens) >= max_tokens:
                break

    if not tokens:
        raise ValueError(
            f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=..."
        )

    return tokens


# =========================
# GitHub client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None


class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "executed-step-fetcher/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())

        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                return i

            if st.reset_epoch and st.reset_epoch <= now:
                return i

        return 0

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(
        self,
        method: str,
        url: str,
        params: Optional[Dict] = None,
    ) -> Optional[Union[Dict, List]]:

        last_status = None

        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(
                    method,
                    url,
                    params=params,
                    timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                )
            except requests.exceptions.RequestException as e:
                print(f"[request exception] attempt={attempt}: {repr(e)}")
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass

            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                print(f"[not found] 404 {url}")
                return None

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                print(f"[error] {resp.status_code} {url}")
                print(resp.text[:1000])
                return None

            try:
                return resp.json()
            except Exception:
                print(f"[json parse error] {url}")
                return None

        print(
            f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries "
            f"(last_status={last_status})"
        )
        return None


# =========================
# GitHub Actions API helpers
# =========================
def get_run_metadata(gh: GitHubClient, full_name: str, run_id: int) -> Dict:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}"
    data = gh.request_json("GET", url)

    if not data or not isinstance(data, dict):
        raise RuntimeError(
            "Failed to fetch run metadata. Check that RUN_ID is the workflow run ID, not the job ID."
        )

    return data


def get_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    """
    Pull executed jobs and executed steps for a workflow run.
    """
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"

    all_jobs = []
    page = 1

    while True:
        data = gh.request_json(
            "GET",
            url,
            params={
                "per_page": 100,
                "page": page,
            },
        )

        if not data or not isinstance(data, dict):
            break

        jobs = data.get("jobs", [])
        all_jobs.extend(jobs)

        if len(jobs) < 100:
            break

        page += 1

    return all_jobs


def flatten_executed_steps(run: Dict, jobs: List[Dict], full_name: str) -> pd.DataFrame:
    rows = []

    for job in jobs:
        job_id = job.get("id")
        job_name = job.get("name")

        for step in job.get("steps", []):
            rows.append({
                "repo": full_name,
                "run_id": run.get("id"),
                "run_attempt": run.get("run_attempt"),
                "workflow_name": run.get("name"),
                "workflow_path": run.get("path"),
                "run_event": run.get("event"),
                "head_branch": run.get("head_branch"),
                "head_sha": run.get("head_sha"),

                "job_id": job_id,
                "job_name": job_name,
                "job_status": job.get("status"),
                "job_conclusion": job.get("conclusion"),
                "job_started_at": job.get("started_at"),
                "job_completed_at": job.get("completed_at"),
                "job_html_url": job.get("html_url"),

                "step_number": step.get("number"),
                "step_name": step.get("name"),
                "step_status": step.get("status"),
                "step_conclusion": step.get("conclusion"),
                "step_started_at": step.get("started_at"),
                "step_completed_at": step.get("completed_at"),
            })

    return pd.DataFrame(rows)


# =========================
# MAIN
# =========================
def main() -> None:
    tokens = load_tokens_from_env_file(
        TOKENS_ENV_PATH,
        max_tokens=MAX_TOKENS_TO_USE,
    )

    gh = GitHubClient(tokens)

    run = get_run_metadata(gh, FULL_NAME, RUN_ID)
    jobs = get_run_jobs(gh, FULL_NAME, RUN_ID)

    print("\nRun metadata:")
    print("repo:", FULL_NAME)
    print("run_id:", run.get("id"))
    print("workflow:", run.get("name"))
    print("workflow_path:", run.get("path"))
    print("head_sha:", run.get("head_sha"))
    print("status/conclusion:", run.get("status"), "/", run.get("conclusion"))
    print("jobs found:", len(jobs))

    steps_df = flatten_executed_steps(run, jobs, FULL_NAME)

    if steps_df.empty:
        raise RuntimeError("No executed steps found from the jobs API.")

    steps_df.to_csv(OUT_PATH, index=False)

    print("\nSaved executed steps to:")
    print(OUT_PATH)

    print("\nExecuted step names:")
    for _, row in steps_df.sort_values(["job_name", "step_number"]).iterrows():
        print(
            f"job_id={row['job_id']} | "
            f"job={row['job_name']} | "
            f"step_number={row['step_number']} | "
            f"step={row['step_name']} | "
            f"{row['step_started_at']} -> {row['step_completed_at']}"
        )


if __name__ == "__main__":
    main()


Run metadata:
repo: FooIbar/EhViewer
run_id: 22557070276
workflow: Baseline profile generation
workflow_path: .github/workflows/baseline-profile.yml
head_sha: 66e43924bddcc773606aa231dc6a8fc4ccea5c76
status/conclusion: completed / success
jobs found: 1

Saved executed steps to:
C:\Android Mobile App\ICST2026_Ext\log\executed_steps_run_22557070276.csv

Executed step names:
job_id=65336353665 | job=baseline-profile | step_number=1 | step=Set up job | 2026-03-02T00:55:04Z -> 2026-03-02T00:55:07Z
job_id=65336353665 | job=baseline-profile | step_number=2 | step=Free disk space | 2026-03-02T00:55:07Z -> 2026-03-02T00:55:34Z
job_id=65336353665 | job=baseline-profile | step_number=3 | step=Enable KVM group perms | 2026-03-02T00:55:34Z -> 2026-03-02T00:55:34Z
job_id=65336353665 | job=baseline-profile | step_number=4 | step=Checkout | 2026-03-02T00:55:34Z -> 2026-03-02T00:55:35Z
job_id=65336353665 | job=baseline-profile | step_number=5 | step=Setup Java | 2026-03-02T00:55:35Z -> 2026-03-02T00:5